In [ ]:
%pip install sentence-transformers

In [ ]:
# !pip install giotto-tda sentence-transformers pandas scikit-learn nltk

import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceLandscape

# Pobranie tokenizatora zdań
nltk.download('punkt')

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
# 1. Wczytanie danych (załóżmy, że masz plik 'news_dataset.csv')
# df = pd.read_csv('news_dataset.csv')

# DO TESTÓW: Tworzymy mały zbiór zabawek (odkomentuj powyższe dla pełnych danych)
df = pd.DataFrame({
    'text': [
        "The quick brown fox jumps over the lazy dog. It was a sunny day.", 
        "Aliens have landed in New York. They are giving away free pizza. The government is hiding it.",
        "Stock markets closed higher today. Investors are optimistic about the upcoming quarter.",
        "Drinking bleach cures all diseases instantly! Doctors hate this one simple trick. Buy now."
    ],
    'label': [1, 0, 1, 0] # 1 - Real, 0 - Fake
})

# Inicjalizacja modelu językowego
model = SentenceTransformer('all-MiniLM-L6-v2')
pca = PCA(n_components=3) # Redukcja wymiaru dla wydajności TDA

def text_to_point_cloud(text):
    sentences = sent_tokenize(str(text))
    if len(sentences) < 2:
        # Jeśli tekst jest za krótki, dodajemy sztuczne zdanie/zaburzenie, by stworzyć przestrzeń
        sentences.append(text + " additional context.")
        
    embeddings = model.encode(sentences)
    
    # Redukcja wymiaru do 3D
    if len(embeddings) >= 3:
        point_cloud = pca.fit_transform(embeddings)
    else:
        # Fallback jeśli zdań jest mało (PCA wymaga n_samples >= n_components)
        point_cloud = embeddings[:, :3] 
        
    return point_cloud

print("Generowanie chmur punktów...")
# Dla 45 tys. artykułów to zajmie trochę czasu! Warto zrobić to na mniejszej próbce.
point_clouds = [text_to_point_cloud(text) for text in df['text']]
y = df['label'].values

In [ ]:
print("Obliczanie homologii persystentnych i krajobrazów...")

# Konfiguracja Vietoris-Rips
# homology_dimensions=[0, 1] oznacza, że patrzymy na H0 i H1
vr = VietorisRipsPersistence(homology_dimensions=[0, 1], n_jobs=-1)

# Obliczenie diagramów persystencji (zwraca tablice 3D)
diagrams = vr.fit_transform(point_clouds)

# Konfiguracja Persistent Landscapes
# n_layers określa ile poziomów krajobrazu bierzemy pod uwagę
# n_bins to rozdzielczość wektora
pl = PersistenceLandscape(n_layers=5, n_bins=50)

# Przekształcenie diagramów w płaskie wektory (features)
X_topological = pl.fit_transform(diagrams)

print(f"Kształt macierzy cech topologicznych: {X_topological.shape}")

In [ ]:
# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X_topological, y, test_size=0.2, random_state=42)

# Inicjalizacja i trening klasyfikatora (Random Forest dobrze radzi sobie z cechami TDA)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predykcja i ewaluacja
y_pred = clf.predict(X_test)

print("Raport klasyfikacji na podstawie cech topologicznych:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

print("Uruchamianie naiwnego baseline'u (TF-IDF)...")

# 1. Wektoryzacja tekstu (zamiana na macierz częstości słów)
# Używamy max_features, aby ograniczyć wymiarowość
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_naive = vectorizer.fit_transform(df['text'])

# 2. Podział na zbiór treningowy i testowy
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_naive, df['label'], test_size=0.2, random_state=42
)

# 3. Szybki trening klasyfikatora
clf_naive = LogisticRegression(max_iter=1000)
clf_naive.fit(X_train_n, y_train_n)

# 4. Predykcja i ewaluacja
y_pred_n = clf_naive.predict(X_test_n)

print("Raport klasyfikacji dla naiwnego podejścia (TF-IDF + Logistic Regression):")
print(classification_report(y_test_n, y_pred_n))

Uruchamianie naiwnego baseline'u (TF-IDF)...


NameError: name 'df' is not defined